In [1]:
# Import required packages
import pandas as pd
import numpy as np
import os
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = os.path.join(PROJECT_DIR, "Data")
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "merged_master.pkl")

In [3]:
data = pd.read_pickle(INPUT_DATA)

In [4]:
# Prepare features and target
# Target variable
TARGET = 'f_cumret1'

# Select features
# Exclude non-feature columns and potential other targets
NON_FEATURES = ['date', 'ticker', 'permno', 'shrout', 'prc'] 

# Identify numeric columns
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

# Filter features
FEATURES = []
for c in numeric_cols:
    if c == TARGET:
        continue
    if c in NON_FEATURES:
        continue
    # Exclude other return/abnormal return columns to prevent leakage
    # Assuming targets start with 'ret' or 'ar_'
    if c.startswith('ret') or c.startswith('ar_'):
        continue
    FEATURES.append(c)

# Remove missing values
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Number of features: {len(FEATURES)}")
print(f"Features: {FEATURES}")

Sample size: 16,743,676
Target: f_cumret1
Number of features: 31
Features: ['net_sentiment', 'extreme_bullish_80', 'extreme_bullish_90', 'extreme_bearish_80', 'extreme_bearish_90', 'disagreement_index', 'raw_volume', 'log_volume', 'unique_user_count', 'volume_diff', 'log_volume_change', 'abn_volume_5d', 'abn_attention_std_5d', 'attention_surge_5d', 'abn_volume_21d', 'abn_attention_std_21d', 'attention_surge_21d', 'abn_volume_63d', 'abn_attention_std_63d', 'attention_surge_63d', 'abn_volume_250d', 'abn_attention_std_250d', 'attention_surge_250d', 'silence_gap_hours', 'relative_volume', 'attention_hhi', 'abnormal_sentiment_1d', 'abnormal_sentiment_5d', 'abnormal_sentiment_21d', 'abnormal_sentiment_63d', 'abnormal_sentiment_250d']


# In-Sample Neural Network

In [ ]:
# Neural network parameters
nn_params = {
    'hidden_layer_sizes': (128, 64, 32),
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,
    'batch_size': 'auto',
    'learning_rate': 'adaptive',
    'learning_rate_init': 0.001,
    'max_iter': 200,
    'random_state': 42,
    'early_stopping': True,
    'validation_fraction': 0.1,
    'n_iter_no_change': 10,
    'verbose': False
}

# Parallel processing settings (for OOS predictions)
N_JOBS = 16  # Use all available CPU cores (-1), or set to specific number

# In-sample training window (adjustable)
TRAIN_START_DATE = '2011-01-01'
TRAIN_END_DATE = '2011-12-31'

# Add date column and sort
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])
model_data = model_data.sort_values('date')
model_data['year_month'] = model_data['date'].dt.to_period('M')

# Filter data to training window
train_start = pd.to_datetime(TRAIN_START_DATE)
train_end = pd.to_datetime(TRAIN_END_DATE)
train_mask = (model_data['date'] >= train_start) & (model_data['date'] <= train_end)

X_train = model_data.loc[train_mask, FEATURES]
y_train = model_data.loc[train_mask, TARGET]

print(f"Training window: {TRAIN_START_DATE} to {TRAIN_END_DATE}")
print(f"Training samples: {len(X_train):,}")
print(f"Number of features: {len(FEATURES)}")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train model
import time
start_time = time.time()

nn_model = MLPRegressor(**nn_params)
nn_model.fit(X_train_scaled, y_train)

elapsed_time = time.time() - start_time

# Make in-sample predictions
y_pred = nn_model.predict(X_train_scaled)

# Evaluate performance
r2 = r2_score(y_train, y_pred)
mse = mean_squared_error(y_train, y_pred)
rmse = np.sqrt(mse)

print(f"\nIn-Sample Neural Network Results")
print("=" * 50)
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print(f"\nModel Architecture: {nn_params['hidden_layer_sizes']}")
print(f"Number of features: {len(FEATURES)}")
print(f"Training samples: {len(y_train):,}")
print(f"Training time: {elapsed_time:.2f} seconds")

# OOS Predictions (Monthly Training, Daily Predictions)

In [ ]:
# Out-of-sample predictions with MONTHLY TRAINING but DAILY PREDICTIONS
# The model is trained once per month (at month-end) and used to predict all days in the following month

# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW = 252  # Rolling window size in trading days (252 = one year)

# Neural network parameters for OOS (same as in-sample)
nn_params_oos = nn_params.copy()

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

# Get unique months in the OOS period
oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Rolling window: {WINDOW} trading days")
print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")
print(f"Number of features: {len(FEATURES)}")
print(f"Parallel processing: {N_JOBS} cores")

In [ ]:
def train_and_predict_month(month_idx, pred_month, model_data, unique_dates, oos_dates, 
                            nn_params, FEATURES, TARGET, WINDOW):
    """
    Train model for a single month and generate predictions for all days in that month.
    This function is designed to be run in parallel.
    """
    predictions = []
    
    # Get all prediction dates in this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    
    if len(month_dates) == 0:
        return predictions
    
    # Training cutoff: end of the previous month (first day of pred_month - 1 day)
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)
    
    # Find the last trading day before or on train_cutoff
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return predictions
    last_train_date = train_dates[-1]
    
    # ROLLING WINDOW: Train on last WINDOW trading days before the month
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx >= WINDOW:
        start_date = unique_dates[last_train_date_idx - WINDOW + 1]
        train_mask = (model_data['date'] >= start_date) & (model_data['date'] <= last_train_date)
        X_train = model_data.loc[train_mask, FEATURES]
        y_train = model_data.loc[train_mask, TARGET]
        
        if len(X_train) > 0:
            # Standardize features
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            
            nn_model = MLPRegressor(**nn_params)
            nn_model.fit(X_train_scaled, y_train)
            
            # Use this model to predict for all days in the month
            for pred_date in month_dates:
                test_mask = model_data['date'] == pred_date
                X_test = model_data.loc[test_mask, FEATURES]
                
                if len(X_test) == 0:
                    continue
                
                test_indices = model_data.index[test_mask]
                X_test_scaled = scaler.transform(X_test)
                y_pred = nn_model.predict(X_test_scaled)
                
                for idx, pred in zip(test_indices, y_pred):
                    predictions.append({
                        'date': pred_date,
                        'index': idx,
                        'prediction': pred
                    })
    
    return predictions

In [ ]:
# Run parallel processing across months
print(f"Starting parallel processing with {N_JOBS} jobs...")
print(f"Processing {len(oos_months)} months...")

# Execute in parallel
all_results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(train_and_predict_month)(
        month_idx, pred_month, model_data, unique_dates, oos_dates,
        nn_params_oos, FEATURES, TARGET, WINDOW
    )
    for month_idx, pred_month in enumerate(oos_months)
)

# Aggregate results
predictions = []
for result in all_results:
    predictions.extend(result)

print(f"\nCompleted.")
print(f"Total predictions: {len(predictions):,}")

In [ ]:
# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'prediction']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"Non-null predictions: {predictions_df['prediction'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

In [ ]:
# Save predictions to Data directory
OUTPUT_DATA_DIR = os.path.join(PROJECT_DIR, "Data")
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DATA_DIR, "predictions_neural_network_all_features.pkl")
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")